    # EDA DisasterWatch

    Exploratory Data Analysis data bencana BNPB periode **1 Januari 2026 sampai 21 Oktober 2026**.
    Notebook ini membaca data hasil download dari `data/processed/`.
    

In [ ]:
    from pathlib import Path
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from IPython.display import display
    
    sns.set_theme(style="whitegrid")
    pd.set_option("display.max_columns", 120)
    ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
    DATA = sorted((ROOT / "data" / "processed").glob("bnpb_bencana_*.csv"))[-1]
    df = pd.read_csv(DATA, encoding="utf-8-sig")
    print(df.shape)
    df.head()
    

## 1. Struktur dan kualitas data

In [ ]:
    df.info()
    display(df.describe(include="all").T)
    missing = (df.isna().mean().mul(100).sort_values(ascending=False)
               .to_frame("missing_pct"))
    display(missing.head(25))
    

In [ ]:
    date_cols = ["tahun", "bulan", "tanggal"]
    df[date_cols] = df[date_cols].apply(pd.to_numeric, errors="coerce")
    df["date"] = pd.to_datetime(dict(year=df.tahun, month=df.bulan, day=df.tanggal), errors="coerce")
    print("Tanggal:", df.date.min(), "s.d.", df.date.max())
    print("Duplikat baris penuh:", df.duplicated().sum())
    

## 2. Distribusi kejadian

In [ ]:
    display(df["kejadian"].value_counts().to_frame("jumlah"))
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.countplot(data=df, y="kejadian", order=df.kejadian.value_counts().index, ax=ax)
    ax.set(title="Jumlah laporan berdasarkan jenis bencana", xlabel="Jumlah laporan", ylabel="")
    plt.tight_layout()
    

In [ ]:
    monthly = df.groupby(df.date.dt.to_period("M")).size().rename("jumlah").reset_index()
    monthly["date"] = monthly.date.dt.to_timestamp()
    ax = monthly.plot(x="date", y="jumlah", marker="o", figsize=(11, 4), legend=False)
    ax.set(title="Tren laporan bencana per bulan", xlabel="Bulan", ylabel="Jumlah laporan")
    plt.tight_layout()
    

In [ ]:
    top_prov = df.provinsi.value_counts().head(15).sort_values()
    top_prov.plot.barh(figsize=(9, 6), title="15 provinsi dengan laporan terbanyak")
    plt.xlabel("Jumlah laporan")
    plt.tight_layout()
    

## 3. Dampak bencana dan koordinat

In [ ]:
    impact_cols = ["meninggal", "hilang", "terluka", "menderita", "mengungsi",
                   "rumah_rusak", "terendam", "lahan"]
    impact = df[impact_cols].apply(pd.to_numeric, errors="coerce").sum().sort_values(ascending=False)
    display(impact.to_frame("total"))
    print("Koordinat tersedia:", df[["geometry_lon", "geometry_lat"]].notna().all(axis=1).mean())
    

## 4. Kualitas data dan potensi duplikasi

In [ ]:
    # Selain duplikasi penuh, cek indikasi laporan ganda berdasarkan kombinasi inti.
    key_cols = ["tahun", "bulan", "tanggal", "id_kab", "kejadian", "lokasi"]
    duplicate_groups = (df.groupby(key_cols, dropna=False).size()
                          .reset_index(name="jumlah_baris")
                          .query("jumlah_baris > 1")
                          .sort_values("jumlah_baris", ascending=False))
    print("Duplikasi baris penuh:", df.duplicated().sum())
    print("Grup indikasi laporan ganda:", len(duplicate_groups))
    print("Baris yang berada dalam grup ganda:", duplicate_groups.jumlah_baris.sum() if len(duplicate_groups) else 0)
    display(duplicate_groups.head(10))
    

## 5. Tren waktu yang lebih dalam

In [ ]:
    # Heatmap jumlah laporan: bulan x jenis bencana.
    df["year_month"] = df.date.dt.to_period("M").astype(str)
    type_month = pd.crosstab(df["year_month"], df["kejadian"])
    plt.figure(figsize=(14, 6))
    sns.heatmap(type_month, annot=True, fmt="g", cmap="YlOrRd")
    plt.title("Heatmap laporan per bulan dan jenis bencana")
    plt.xlabel("Jenis bencana"); plt.ylabel("Bulan")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    display(type_month)
    

In [ ]:
    # Perubahan bulanan relatif terhadap bulan sebelumnya.
    monthly_type = (df.groupby(["year_month", "kejadian"]).size()
                      .unstack(fill_value=0).sort_index())
    monthly_change = monthly_type.pct_change().replace([np.inf, -np.inf], np.nan) * 100
    display(monthly_change.round(1).tail(5))
    

## 6. Konsentrasi wilayah

In [ ]:
    province_counts = df["provinsi"].value_counts().rename("jumlah_laporan").to_frame()
    province_counts["persentase"] = province_counts.jumlah_laporan / len(df) * 100
    province_counts["persentase_kumulatif"] = province_counts.persentase.cumsum()
    display(province_counts.head(15).round(2))
    top5_share = province_counts.head(5).persentase.sum()
    top10_share = province_counts.head(10).persentase.sum()
    print(f"Top 5 provinsi menyumbang {top5_share:.1f}% laporan")
    print(f"Top 10 provinsi menyumbang {top10_share:.1f}% laporan")
    

In [ ]:
    top_province_type = (pd.crosstab(df.provinsi, df.kejadian)
                           .loc[province_counts.head(10).index])
    plt.figure(figsize=(13, 7))
    top_province_type.plot(kind="bar", stacked=True, figsize=(13, 7), colormap="tab20")
    plt.title("Komposisi jenis bencana pada 10 provinsi teratas")
    plt.xlabel(""); plt.ylabel("Jumlah laporan"); plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    

## 7. Group by masing-masing jenis bencana

In [ ]:
    # Ringkasan utama untuk setiap jenis bencana.
    disaster_groupby = df.groupby("kejadian", dropna=False).agg(
        jumlah_laporan=("kejadian", "size"),
        jumlah_kejadian=("jumlah_kejadian", "sum"),
        provinsi_terdampak=("provinsi", "nunique"),
        kabkot_terdampak=("kabkot", "nunique"),
        meninggal=("meninggal", "sum"),
        hilang=("hilang", "sum"),
        terluka=("terluka", "sum"),
        menderita=("menderita", "sum"),
        mengungsi=("mengungsi", "sum"),
        rumah_rusak=("rumah_rusak", "sum"),
        terendam=("terendam", "sum"),
        lahan_ha=("lahan", "sum"),
    ).sort_values("jumlah_laporan", ascending=False)
    display(disaster_groupby.round(2))
    

In [ ]:
    # Group by jenis bencana dan provinsi: wilayah dominan untuk setiap bencana.
    disaster_province = (df.groupby(["kejadian", "provinsi"], dropna=False)
                           .agg(jumlah_laporan=("kejadian", "size"),
                                meninggal=("meninggal", "sum"),
                                mengungsi=("mengungsi", "sum"),
                                rumah_rusak=("rumah_rusak", "sum"),
                                terendam=("terendam", "sum"))
                           .reset_index()
                           .sort_values(["kejadian", "jumlah_laporan"], ascending=[True, False]))
    display(disaster_province.head(20))

    # Tampilkan top-5 provinsi secara terpisah untuk setiap jenis bencana.
    for disaster, group in disaster_province.groupby("kejadian", sort=True):
        print(f"\n=== {disaster} ===")
        display(group.head(5).reset_index(drop=True))
    

In [ ]:
    # Group by jenis bencana dan bulan untuk melihat pola waktunya.
    disaster_month = (df.groupby(["kejadian", "year_month"], dropna=False)
                        .agg(jumlah_laporan=("kejadian", "size"),
                             jumlah_kejadian=("jumlah_kejadian", "sum"),
                             meninggal=("meninggal", "sum"),
                             mengungsi=("mengungsi", "sum"),
                             rumah_rusak=("rumah_rusak", "sum"))
                        .reset_index())
    display(disaster_month.sort_values(["kejadian", "year_month"]).head(30))
    

## 8. Severity dan dampak per kejadian

In [ ]:
    numeric_impact = ["meninggal", "hilang", "terluka", "menderita", "mengungsi",
                      "rumah_rusak", "terendam", "lahan"]
    for col in numeric_impact:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df["fatality_or_injury"] = df["meninggal"] + df["hilang"] + df["terluka"]
    df["people_affected"] = df["menderita"] + df["mengungsi"]
    df["has_human_impact"] = (df["fatality_or_injury"] + df["people_affected"] > 0).astype(int)
    severity_by_type = df.groupby("kejadian").agg(
        laporan=("kejadian", "size"),
        total_korban=("fatality_or_injury", "sum"),
        total_terdampak=("people_affected", "sum"),
        total_rumah_rusak=("rumah_rusak", "sum"),
        median_korban=("fatality_or_injury", "median"),
        persentase_berdampak=("has_human_impact", "mean"),
    ).sort_values("total_terdampak", ascending=False)
    severity_by_type["persentase_berdampak"] *= 100
    display(severity_by_type.round(2))
    

In [ ]:
    # Laporan paling berat menurut beberapa dimensi dampak.
    display(df.nlargest(10, "fatality_or_injury")[["date", "provinsi", "kabkot", "kejadian", "fatality_or_injury", "kronologis"]])
    display(df.nlargest(10, "people_affected")[["date", "provinsi", "kabkot", "kejadian", "people_affected", "kronologis"]])
    

## 8. Korelasi dan pola dampak

In [ ]:
    corr_cols = ["meninggal", "hilang", "terluka", "menderita", "mengungsi", "rumah_rusak", "terendam", "lahan"]
    corr = df[corr_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
    plt.title("Korelasi antar indikator dampak")
    plt.tight_layout()
    display(corr.round(2))
    

## 9. Analisis spasial sederhana

In [ ]:
    geo = df.dropna(subset=["geometry_lon", "geometry_lat"]).copy()
    geo["geometry_lon"] = pd.to_numeric(geo.geometry_lon, errors="coerce")
    geo["geometry_lat"] = pd.to_numeric(geo.geometry_lat, errors="coerce")
    geo = geo.dropna(subset=["geometry_lon", "geometry_lat"])
    print(f"Koordinat valid: {len(geo):,} dari {len(df):,} ({len(geo)/len(df)*100:.1f}%)")
    plt.figure(figsize=(12, 7))
    plt.scatter(geo.geometry_lon, geo.geometry_lat, s=8, alpha=.35, c=geo.fatality_or_injury, cmap="Reds")
    plt.colorbar(label="Meninggal + hilang + terluka")
    plt.title("Sebaran koordinat laporan bencana")
    plt.xlabel("Longitude"); plt.ylabel("Latitude")
    plt.tight_layout()
    

## 10. Insight otomatis

In [ ]:
    top_type = df.kejadian.value_counts().idxmax()
    top_type_n = int(df.kejadian.value_counts().max())
    top_prov = df.provinsi.value_counts().idxmax()
    top_prov_n = int(df.provinsi.value_counts().max())
    peak_month = df.year_month.value_counts().idxmax()
    peak_month_n = int(df.year_month.value_counts().max())
    highest_fatality_type = severity_by_type.total_korban.idxmax()
    highest_affected_type = severity_by_type.total_terdampak.idxmax()
    print(f"1. Jenis bencana paling sering: {top_type} ({top_type_n:,} laporan).")
    print(f"2. Provinsi dengan laporan terbanyak: {top_prov} ({top_prov_n:,} laporan).")
    print(f"3. Bulan dengan laporan terbanyak: {peak_month} ({peak_month_n:,} laporan).")
    print(f"4. Jenis dengan total korban jiwa/cedera tertinggi: {highest_fatality_type}.")
    print(f"5. Jenis dengan total penduduk terdampak/pengungsi tertinggi: {highest_affected_type}.")
    print(f"6. Konsentrasi top-5 provinsi: {top5_share:.1f}% dari seluruh laporan.")
    print("Gunakan hasil ini sebagai hipotesis; validasi dengan kualitas pelaporan dan data pembanding sebelum mengambil keputusan.")
    

## Kesimpulan dan pertanyaan analitik

Dokumentasikan temuan penting dari output di atas. Pertanyaan lanjutan yang dapat dijawab: mengapa terjadi lonjakan pada bulan tertentu, apakah konsentrasi wilayah mencerminkan risiko atau kelengkapan pelaporan, jenis bencana mana yang paling berat per laporan, dan bagaimana missing value memengaruhi kesimpulan?